# Step4.2 Time-dependent AUC (Corrected Version)

**Corrections:**
- Replace the binary-classification `plot_DCA` with the survival-analysis-specific dcurves library DCA
- Replace the binary-classification `draw_calibration` with the IPCW calibration curve
- Output is saved to the `result1/` and `img1/` directories

In [ ]:
#pip install dcurves

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from onekey_algo import get_param_in_cwd
from onekey_algo.custom.components.metrics import get_time_dependent_gt
from onekey_algo.custom.components.comp1 import normalize_df, merge_results, draw_roc
from onekey_algo.custom.components.metrics import analysis_pred_binary
from lifelines import KaplanMeierFitter
from dcurves import dca, plot_graphs
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('result1', exist_ok=True)
os.makedirs('img1', exist_ok=True)

In [ ]:
# ==================== KM-based calibration curve function ====================
def plot_calibration_km(pred_list, data, model_names, event_col, duration_col,
                        target_time, n_bins=5, title='Calibration Curve'):
    """
    Survival-analysis calibration curve (KM-based, auto-detect score direction)
    
    Method: manual binning (pd.qcut) + KaplanMeier estimate (at the target time point)
    Direction: auto-detect the correlation between pred and event, uniformly convert to survival probability
    
    Handling of skewed distributions:
    - If the score distribution is extremely skewed (IQR/range < 0.1) or range-compressed (range < 0.3), apply a quantile transform
    - Map the scores to a [0,1] uniform distribution before binning, to ensure valid calibration-curve bins
    
    X-axis: Mean Predicted Survival Probability
    Y-axis: Observed Survival Probability (KM estimate at target_time)
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = plt.cm.tab10(np.linspace(0, 1, len(model_names)))
    
    time_arr = data[duration_col].values.astype(float)
    event_arr = data[event_col].values.astype(float)
    
    for idx, (pred, name) in enumerate(zip(pred_list, model_names)):
        pred_arr = np.array(pred, dtype=float)
        
        # Detect skewness or range compression
        # Condition 1: IQR/range < 0.1 -> extremely skewed (outliers dominate range)
        # Condition 2: p_range < 0.3 -> after global minmax this subset occupies only a small part of [0,1]
        p_range = pred_arr.max() - pred_arr.min()
        if p_range > 0:
            iqr = np.percentile(pred_arr, 75) - np.percentile(pred_arr, 25)
            skew_ratio = iqr / p_range
        else:
            skew_ratio = 1.0
        
        if skew_ratio < 0.1 or p_range < 0.3:
            # Extremely skewed or range-compressed: use a quantile transform to map to a [0,1] uniform distribution
            from scipy.stats import rankdata
            pred_uniform = (rankdata(pred_arr) - 1) / (len(pred_arr) - 1)
            print(f'  Calibration [{name}]: skew_ratio={skew_ratio:.4f}, p_range={p_range:.4f} -> rank-based transform')
        else:
            pred_uniform = pred_arr.copy()
        
        # Auto-detect direction: pred positively correlated with event -> risk score -> flip to survival probability
        corr = np.corrcoef(pred_uniform, event_arr)[0, 1]
        if corr > 0:
            pred_surv = 1.0 - pred_uniform
        else:
            pred_surv = pred_uniform.copy()
        pred_surv = np.clip(pred_surv, 0.001, 0.999)
        
        # Bin by predicted survival probability (pd.qcut keeps each group balanced in size)
        try:
            bin_labels = pd.qcut(pred_surv, q=n_bins, labels=False, duplicates='drop')
        except Exception:
            continue
        
        mean_predicted, observed_km = [], []
        for b in sorted(np.unique(bin_labels)):
            mask = bin_labels == b
            if mask.sum() < 5:
                continue
            
            # X-axis: mean predicted survival probability of this group
            mean_predicted.append(float(pred_surv[mask].mean()))
            
            # Y-axis: KM estimate of the survival rate of this group at target_time
            kmf = KaplanMeierFitter()
            try:
                kmf.fit(time_arr[mask], event_observed=event_arr[mask])
                surv_at_t = kmf.survival_function_at_times(target_time).values[0]
                observed_km.append(float(surv_at_t))
            except Exception:
                observed_km.append(1.0 - float(event_arr[mask].mean()))
        
        if len(mean_predicted) > 0:
            ax.plot(mean_predicted, observed_km, 'o-', color=colors[idx],
                    label=name, linewidth=2, markersize=8)
    
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration', alpha=0.6)
    ax.set_xlabel('Predicted Survival Probability', fontsize=12)
    ax.set_ylabel('Observed Survival Probability (KM)', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.legend(loc='lower right')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig, ax

In [ ]:
# ==================== DCA function (time-dependent label) ====================
def plot_dca(pred_list, model_names, data, event_col, title='DCA'):
    """
    DCA curve - use the time-dependent label as the outcome
    
    Core logic:
    - Outcome = data['label'] (generated by get_time_dependent_gt, 1=death within target time, 0=alive/censored)
    - Predictor = predicted death-risk probability (auto-detect and align direction)
    - If pred is negatively correlated with label (i.e. pred is the survival direction), flip it to 1-pred
    
    Note: do not use the global event_col, to avoid long-term death cases contaminating the short-term analysis
    """
    # Use the time-dependent label (1=recurrence/death within the target time)
    label_arr = data['label'].values.astype(float)
    prevalence = label_arr.mean()
    
    # Dynamically adjust the Y-axis and threshold range according to the event rate
    y_max = min(max(prevalence + 0.15, 0.4), 1.0)
    thresh_max = min(max(0.5, prevalence + 0.2), 1.0)
    
    dca_data = pd.DataFrame()
    dca_data['outcome'] = label_arr
    
    valid_names = []
    for name, pred in zip(model_names, pred_list):
        pred_arr = np.array(pred, dtype=float)
        # Auto-detect direction: correlation between pred and the time-dependent label
        corr = np.corrcoef(pred_arr, label_arr)[0, 1]
        if corr < 0:
            # pred negatively correlated with death -> pred is the survival direction -> flip to death risk
            pred_risk = 1.0 - pred_arr
        else:
            # pred positively correlated with death -> already the risk direction
            pred_risk = pred_arr.copy()
        dca_data[name] = np.clip(pred_risk, 0, 1)
        valid_names.append(name)
    
    if len(valid_names) == 0:
        print("Warning: No valid predictors")
        return None
    
    try:
        dca_results = dca(
            data=dca_data,
            outcome='outcome',
            modelnames=valid_names,
            thresholds=np.arange(0, thresh_max, 0.01)
        )
        plot_graphs(plot_df=dca_results, graph_type='net_benefit', y_limits=[-0.05, y_max])
        fig = plt.gcf()
        fig.set_size_inches(10, 6)
        plt.title(title)
        plt.tight_layout()
        return fig
    except Exception as e:
        print(f"dcurves failed ({e}), using fallback...")
        fig, ax = plt.subplots(figsize=(10, 6))
        thresholds = np.arange(0.01, thresh_max, 0.01)
        n = len(label_arr)
        
        for name in valid_names:
            prob = dca_data[name].values
            net_benefits = []
            for t in thresholds:
                tp = np.sum((prob >= t) & (label_arr == 1))
                fp = np.sum((prob >= t) & (label_arr == 0))
                nb = (tp / n) - (fp / n) * (t / (1 - t))
                net_benefits.append(nb)
            ax.plot(thresholds, net_benefits, label=name, linewidth=2)
        
        all_benefits = [prevalence - (1-prevalence)*(t/(1-t)) for t in thresholds]
        ax.plot(thresholds, all_benefits, 'k--', label='Treat All', alpha=0.7)
        ax.axhline(y=0, color='gray', linestyle=':', label='Treat None', alpha=0.7)
        
        ax.set_xlabel('Threshold Probability')
        ax.set_ylabel('Net Benefit')
        ax.set_title(title)
        ax.legend(loc='upper right')
        ax.set_ylim([-0.05, y_max])
        plt.tight_layout()
        return fig

In [ ]:
# ==================== Main analysis code ====================
event_col = get_param_in_cwd('event_col')
duration_col = get_param_in_cwd('duration_col')
model_names = [mn for mn in get_param_in_cwd('summary_models')]

joinit_info = pd.read_csv('results/joinit_info.csv', dtype={'ID':str})
joinit_info['ID'] = joinit_info['ID'].astype(str)

# Standardization: exclude the ID, event, time and group columns
joinit_info = normalize_df(joinit_info, not_norm=['ID', event_col, duration_col, 'group'], method='minmax')

print(f'Data shape: {joinit_info.shape}')
print(f'Models: {model_names}')

In [ ]:
metric = []
youden = {}

for tname, time in get_param_in_cwd('time_settings').items():
    for subset in get_param_in_cwd('subsets'):
        data = joinit_info[joinit_info['group'] == subset]
        time_label_data = get_time_dependent_gt(data, time=time, event_col=event_col, duration_col=duration_col)
        
        if len(np.unique(time_label_data['label'])) == 1:
            print(f'Skip {tname} {subset}: only one class')
            continue
        
        time_label_data['ID'] = time_label_data['ID'].astype(str)
        ALL_results = merge_results(data, time_label_data)
        ALL_results = ALL_results.drop_duplicates('ID')
        print(f'{tname} {subset}: {ALL_results.shape}')
        ALL_results.to_csv(f'result1/time_roc_{tname}_{subset}.csv', index=False)
        
        gt = [1-np.array(ALL_results['label']) for _ in model_names]
        pred_train = [np.array(ALL_results[d]) for d in model_names]
        
        # 1. ROC curve
        draw_roc(gt, pred_train, labels=model_names, title=f'Cohort {subset} {tname} ROC')
        plt.savefig(f'img1/{tname}_{subset}_auc.svg')
        plt.close()  # close to avoid popup window
        
        # 2. DCA (use the time-dependent label data['label'])
        try:
            fig = plot_dca(pred_train, model_names,
                          data=ALL_results, event_col=event_col,
                          title=f'Cohort {subset} {tname} DCA')
            if fig:
                plt.savefig(f'img1/{tname}_{subset}_dca.svg')
                plt.close()
        except Exception as e:
            print(f'DCA failed: {e}')
        
        # 3. KM-based calibration curve (auto direction detection + KM@target time point)
        try:
            
            fig, ax = plot_calibration_km(
                pred_train, ALL_results, model_names,
                event_col, duration_col, target_time=time,
                n_bins=5, title=f'Cohort {subset} {tname} Calibration')
            plt.savefig(f'img1/{tname}_{subset}_cali.svg')
            plt.close()
        except Exception as e:
            print(f'Calibration failed: {e}')
        
        # Compute metrics
        for mname, y, score in zip(model_names, gt, pred_train):
            acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y, score)
            ci_str = f'{ci[0]:.4f} - {ci[1]:.4f}'
            metric.append((mname, acc, auc, ci_str, tpr, tnr, ppv, npv, tname, f'{subset[0].upper() + subset[1:]}'))
            youden[mname] = thres

time_metrics = pd.DataFrame(metric, columns=['Signature', 'Accuracy', 'AUC', '95% CI', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'Survival', 'Cohort'])
time_metrics.to_csv('result1/time_dependent_metrics.csv', index=False)
print('\n=== Time-dependent metrics summary ===')
time_metrics

In [ ]:
print('\n' + '='*60)
print('Step completed (Corrected Version)')
print('='*60)
print('\nCorrections:')
print('  ✓ plot_DCA → dcurves DCA (auto direction alignment)')
print('  ✓ draw_calibration → KM-based calibration curve (binning+KM@target time point)')
print('\nOutput files:')
print('  • result1/time_dependent_metrics.csv')
print('  • img1/*.svg')